# V-JEPA 2 + Decoder — Ball Occlusion Inference

**Run on Kaggle T4 GPU.**  Requires:
- Kaggle dataset `ball-clips` uploaded from `data/processed/ball_clips/` on your Mac
- Kaggle Secret `HF_TOKEN` set to your HuggingFace token (V-JEPA 2 weights + decoder download automatically)

Pipeline per video clip:
```
16 frames [B,3,16,256,256]  (ball clips stored at 384px — resized to 256 on load)
  Spatial mask: patch columns 5–10 excluded from encoder (pixels 80–176)
  → V-JEPA 2 Encoder  (visible patches only)  →  Zc  [B, 1280, 1024]
  → V-JEPA 2 Predictor (Zc + target positions) →  Ẑm  [B, 768, 1024]
  Reconstruct full grid:  [B, 2048, 1024]
  For each of 8 temporal positions:
    reshape 256 tokens → [1024, 16, 16]
    → LocalizationDecoder  →  heatmap [1, 256, 256]
```

## 0. Install V-JEPA 2 from Source

In [ ]:
import subprocess, sys
r = subprocess.run('git clone --depth 1 https://github.com/facebookresearch/vjepa2 /tmp/vjepa2 2>&1 || echo already_cloned',
                   shell=True, capture_output=True, text=True)
print(r.stdout)
r = subprocess.run(f'{sys.executable} -m pip install -e /tmp/vjepa2 -q',
                   shell=True, capture_output=True, text=True)
print(r.stdout[-200:] if r.stdout else 'ok')

## 1. Imports & Constants

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from huggingface_hub import hf_hub_download, login

# HuggingFace auth (Kaggle → Add-ons → Secrets → HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)
    print('HuggingFace login OK')
except Exception as e:
    print(f'HF login skipped ({e})')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    cap = torch.cuda.get_device_capability()
    if cap[0] < 7:
        print(f'WARNING: GPU capability {cap} < 7.0 — switch to T4'); DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')

# V-JEPA 2 ViT-L/16 @ 256
IMAGE_SIZE   = 256
PATCH_SIZE   = 16
PATCH_GRID   = IMAGE_SIZE // PATCH_SIZE    # 16
EMBED_DIM    = 1024
NUM_FRAMES   = 16
TUBELET_SIZE = 2
N_TEMPORAL   = NUM_FRAMES // TUBELET_SIZE  # 8
N_SPATIAL    = PATCH_GRID * PATCH_GRID     # 256
N_TOTAL      = N_TEMPORAL * N_SPATIAL      # 2048

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

BALL_CLIPS_DIR = Path('/kaggle/input/ball-clips/ball_clips')
OUT_DIR        = Path('/kaggle/working/inference_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Download V-JEPA 2 weights
print('Downloading V-JEPA 2 weights ...')
VJEPA_WEIGHTS = Path(hf_hub_download(
    repo_id='facebook/vjepa2-vitl-fpc64-256',
    filename='original/model.pth',
    local_dir='/kaggle/working',
))
print(f'V-JEPA 2 weights: {VJEPA_WEIGHTS}')

# Download trained decoder
print('Downloading decoder checkpoint ...')
DECODER_CKPT = Path(hf_hub_download(
    repo_id='Bmingg/cs280-localization-decoder-vjepa2',
    filename='decoder_best.pt',
    local_dir='/kaggle/working',
))
print(f'Decoder checkpoint: {DECODER_CKPT}')

print(f'Token layout: {N_TEMPORAL} temporal × {N_SPATIAL} spatial = {N_TOTAL} total')

## 2. Mask Definition

The spatial mask covers patch **columns 5–10** (inclusive) across **all temporal positions**,
corresponding to pixels x∈[80, 176] — the center-right third of the 256×256 frame.

Token index for (temporal=t, row=r, col=c): `t * N_SPATIAL + r * PATCH_GRID + c`

In [ ]:
MASK_COL_START = 5
MASK_COL_END   = 10   # inclusive  (pixels 80–176 out of 256)

context_indices = []
target_indices  = []

for t in range(N_TEMPORAL):
    for r in range(PATCH_GRID):
        for c in range(PATCH_GRID):
            idx = t * N_SPATIAL + r * PATCH_GRID + c
            if MASK_COL_START <= c <= MASK_COL_END:
                target_indices.append(idx)
            else:
                context_indices.append(idx)

ctx_t = torch.tensor(context_indices, dtype=torch.long)
tgt_t = torch.tensor(target_indices,  dtype=torch.long)

n_masked_cols = MASK_COL_END - MASK_COL_START + 1
print(f'Context tokens : {len(context_indices)}  ({N_TEMPORAL} × {PATCH_GRID} × {PATCH_GRID - n_masked_cols})')
print(f'Target  tokens : {len(target_indices)}   ({N_TEMPORAL} × {PATCH_GRID} × {n_masked_cols})')
print(f'Mask pixel region: x=[{MASK_COL_START*PATCH_SIZE}, {(MASK_COL_END+1)*PATCH_SIZE})')

## 3. Load V-JEPA 2 Encoder + Predictor

Weights from `facebook/vjepa2-vitl-fpc64-256` — checkpoint key `target_encoder` (not `encoder`).
V-JEPA 2 predictor outputs **1024-dim** (same space as encoder) — compatible with the decoder.

In [ ]:
print('Loading V-JEPA 2 architecture ...')
result = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_large',
                        pretrained=False, trust_repo=True)
encoder, predictor = result
print(f'encoder  : {type(encoder).__name__}')
print(f'predictor: {type(predictor).__name__}')

print(f'\nLoading weights from {VJEPA_WEIGHTS} ...')
ckpt = torch.load(VJEPA_WEIGHTS, map_location='cpu', weights_only=True)
print(f'Checkpoint keys: {list(ckpt.keys())}')

def load_sd(model, raw_sd):
    sd = {k.replace('module.', ''): v for k, v in raw_sd.items()}
    sd = {k.replace('backbone.', ''): v for k, v in sd.items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert len(missing) == 0 and len(unexpected) == 0, \
        f'missing={missing[:3]}  unexpected={unexpected[:3]}'

enc_key = 'target_encoder' if 'target_encoder' in ckpt else 'encoder'
load_sd(encoder,   ckpt[enc_key])
load_sd(predictor, ckpt['predictor'])

encoder.eval().to(DEVICE)
predictor.eval().to(DEVICE)
for p in list(encoder.parameters()) + list(predictor.parameters()):
    p.requires_grad_(False)

print(f'\nEncoder  : {sum(p.numel() for p in encoder.parameters())/1e6:.0f}M params')
print(f'Predictor: {sum(p.numel() for p in predictor.parameters())/1e6:.0f}M params')

## 4. Load Trained Decoder

In [ ]:
class UpsampleBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class LocalizationDecoder(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.upsample = nn.Sequential(
            UpsampleBlock(embed_dim, 512),
            UpsampleBlock(512, 256),
            UpsampleBlock(256, 128),
            UpsampleBlock(128, 64),
        )
        self.head = nn.Sequential(nn.Conv2d(64, 1, kernel_size=1), nn.Sigmoid())
    def forward(self, x): return self.head(self.upsample(x))

decoder = LocalizationDecoder().to(DEVICE)
ckpt_dec = torch.load(DECODER_CKPT, map_location='cpu', weights_only=True)
decoder.load_state_dict(ckpt_dec['decoder_state_dict'])
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)
print(f'Decoder loaded — val CE={ckpt_dec["val_ce"]:.1f}px  IoU={ckpt_dec["val_iou"]:.3f}  epoch={ckpt_dec["epoch"]}')

## 5. Inference Helpers

In [ ]:
SCALE_384_TO_256 = IMAGE_SIZE / 384.0   # scale GT coords from 384px clips to 256px input


def load_clip(clip_dir: Path, n_frames: int = NUM_FRAMES) -> tuple[torch.Tensor, dict, np.ndarray]:
    """Load frames, resize to IMAGE_SIZE (256), sample to n_frames."""
    with open(clip_dir / 'metadata.json') as f:
        meta = json.load(f)

    frame_paths = sorted((clip_dir / 'frames').glob('*.png'))
    total   = len(frame_paths)
    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    frames  = [np.array(Image.open(frame_paths[i]).resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
                        ).astype(np.float32) / 255.0
               for i in indices]
    video = torch.from_numpy(np.stack(frames)).permute(3, 0, 1, 2).unsqueeze(0)
    return video, meta, indices


def run_vjepa(video: torch.Tensor) -> torch.Tensor:
    """
    Masked encoder + V-JEPA 2 predictor pass.

    Context tokens (cols outside 5–10) are encoded by the encoder.
    Target tokens  (cols 5–10, the masked region) are inferred by the predictor.
    Both are assembled into a full [N_TOTAL, 1024] grid and reshaped to
    [N_TEMPORAL, 1024, 16, 16] for the decoder.

    Args:
        video: [1, 3, T, H, W]  float32 in [0,1]  on CPU
    Returns:
        per_temporal: [N_TEMPORAL, 1024, 16, 16]
    """
    x = (video - IMAGENET_MEAN.unsqueeze(2)) / IMAGENET_STD.unsqueeze(2)
    x = x.to(DEVICE)

    ctx_mask = ctx_t.unsqueeze(0).to(DEVICE)   # [1, N_ctx]
    tgt_mask = tgt_t.unsqueeze(0).to(DEVICE)   # [1, N_tgt]

    with torch.no_grad():
        Zc     = encoder(x, masks=[ctx_mask])                   # [1, N_ctx, 1024]
        Zm_hat = predictor(Zc, [ctx_mask], [tgt_mask])          # [1, N_tgt, 1024]

    Zc     = Zc.squeeze(0).cpu()       # [N_ctx, 1024]
    Zm_hat = Zm_hat.squeeze(0).cpu()   # [N_tgt, 1024]

    full = torch.zeros(N_TOTAL, EMBED_DIM)
    full[ctx_t] = Zc
    full[tgt_t] = Zm_hat

    per_temporal = full.view(N_TEMPORAL, N_SPATIAL, EMBED_DIM).permute(0, 2, 1)
    per_temporal = per_temporal.reshape(N_TEMPORAL, EMBED_DIM, PATCH_GRID, PATCH_GRID)
    return per_temporal


def decode_temporal(per_temporal: torch.Tensor) -> torch.Tensor:
    """Run decoder on each temporal position. Returns [N_TEMPORAL, 1, 256, 256]."""
    heatmaps = []
    with torch.no_grad():
        for t in range(N_TEMPORAL):
            emb = per_temporal[t].unsqueeze(0).to(DEVICE)
            hm  = decoder(emb).squeeze(0).cpu()
            heatmaps.append(hm)
    return torch.stack(heatmaps)   # [N_TEMPORAL, 1, 256, 256]


print('Inference helpers defined.')

## 6. Run Inference on All Clips

In [ ]:
clip_dirs = sorted(BALL_CLIPS_DIR.iterdir())
print(f'Found {len(clip_dirs)} clips: {[d.name for d in clip_dirs]}')

results = {}

for clip_dir in clip_dirs:
    if not clip_dir.is_dir():
        continue
    name = clip_dir.name
    print(f'\n--- {name} ---')

    video, meta, frame_indices = load_clip(clip_dir)
    print(f'  Loaded {meta["n_frames"]} frames, sampled {NUM_FRAMES}  video: {video.shape}')

    per_temporal = run_vjepa(video)             # [8, 1024, 16, 16]
    print(f'  Encoder+Predictor output: {per_temporal.shape}')

    heatmaps = decode_temporal(per_temporal)    # [8, 1, 256, 256]
    print(f'  Heatmaps: {heatmaps.shape}  max={heatmaps.max():.3f}')

    results[name] = {
        'per_temporal':  per_temporal,
        'heatmaps':      heatmaps,
        'meta':          meta,
        'frame_indices': frame_indices,
        'video':         video,
    }

## 7. Visualise — Trajectory Through the Mask

Each column = one temporal position (2 real frames averaged).  
Row 1: raw frame with mask box and GT ball centre (green +).  
Row 2: decoder heatmap.  
Row 3: overlay — heatmap in red channel on top of raw frame.

In [ ]:
MASK_PX_X1 = MASK_COL_START * PATCH_SIZE
MASK_PX_X2 = (MASK_COL_END + 1) * PATCH_SIZE


def heatmap_peak(hm: torch.Tensor) -> tuple[int, int]:
    idx = hm.squeeze().argmax().item()
    return int(idx) % IMAGE_SIZE, int(idx) // IMAGE_SIZE


for name, res in results.items():
    heatmaps     = res['heatmaps']      # [8, 1, 256, 256]
    video        = res['video']         # [1, 3, 16, 256, 256]
    meta         = res['meta']
    frame_idx    = res['frame_indices']

    # Scale GT coords from 384px (stored) to 256px (model input)
    gt_lookup = {b['frame']: (b['cx_384'] * SCALE_384_TO_256, b['cy_384'] * SCALE_384_TO_256)
                 for b in meta['ball_centers'] if b.get('cx_384') is not None}

    fig, axes = plt.subplots(3, N_TEMPORAL, figsize=(3 * N_TEMPORAL, 10))
    fig.suptitle(f'{name}  — V-JEPA 2 predictor + decoder  (mask cols {MASK_COL_START}–{MASK_COL_END})', fontsize=12)

    for t in range(N_TEMPORAL):
        real_frame_slot = t * TUBELET_SIZE
        orig_idx = int(frame_idx[min(real_frame_slot, len(frame_idx)-1)])

        frame_np = video[0, :, real_frame_slot].permute(1, 2, 0).numpy()
        hm_np    = heatmaps[t].squeeze().numpy()
        pred_cx, pred_cy = heatmap_peak(heatmaps[t])

        ax0 = axes[0, t]
        ax0.imshow(frame_np)
        rect = mpatches.Rectangle((MASK_PX_X1, 0), MASK_PX_X2 - MASK_PX_X1, IMAGE_SIZE,
                                   linewidth=1.5, edgecolor='yellow', facecolor='yellow', alpha=0.15)
        ax0.add_patch(rect)
        if orig_idx in gt_lookup:
            gx, gy = gt_lookup[orig_idx]
            ax0.plot(gx, gy, 'g+', ms=12, mew=2, label='GT')
        ax0.plot(pred_cx, pred_cy, 'r+', ms=12, mew=2, label='Pred')
        ax0.set_title(f't={t}  (frame {orig_idx})', fontsize=7)
        ax0.axis('off')
        if t == 0:
            ax0.set_ylabel('Raw + mask', fontsize=8)

        ax1 = axes[1, t]
        ax1.imshow(hm_np, cmap='hot', vmin=0, vmax=1)
        ax1.axis('off')
        if t == 0:
            ax1.set_ylabel('Decoder\nheatmap', fontsize=8)

        ax2 = axes[2, t]
        overlay = frame_np.copy()
        overlay[:, :, 0] = np.clip(overlay[:, :, 0] + hm_np * 0.7, 0, 1)
        ax2.imshow(overlay)
        ax2.add_patch(mpatches.Rectangle((MASK_PX_X1, 0), MASK_PX_X2 - MASK_PX_X1, IMAGE_SIZE,
                                         linewidth=1.5, edgecolor='yellow', facecolor='none'))
        ax2.axis('off')
        if t == 0:
            ax2.set_ylabel('Overlay', fontsize=8)

    plt.tight_layout()
    fig.savefig(OUT_DIR / f'{name}_trajectory.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUT_DIR}/{name}_trajectory.png')

## 8. Quantitative Summary

Centre Error between predicted heatmap peak and GT ball position,
split by whether the ball is **inside** or **outside** the mask.
The inside-mask rows are the actual occlusion test.

In [ ]:
for name, res in results.items():
    heatmaps  = res['heatmaps']
    meta      = res['meta']
    frame_idx = res['frame_indices']

    gt_lookup = {b['frame']: (b['cx_384'] * SCALE_384_TO_256, b['cy_384'] * SCALE_384_TO_256)
                 for b in meta['ball_centers'] if b.get('cx_384') is not None}

    ce_inside, ce_outside = [], []

    for t in range(N_TEMPORAL):
        orig_idx = int(frame_idx[min(t * TUBELET_SIZE, len(frame_idx)-1)])
        if orig_idx not in gt_lookup:
            continue
        gx, gy   = gt_lookup[orig_idx]
        px, py   = heatmap_peak(heatmaps[t])
        ce       = math.sqrt((px - gx)**2 + (py - gy)**2)
        in_mask  = MASK_PX_X1 <= gx < MASK_PX_X2
        (ce_inside if in_mask else ce_outside).append(ce)

    def _fmt(lst):
        return f'{np.mean(lst):.1f}px  (n={len(lst)})' if lst else 'no GT'

    print(f'{name}:')
    print(f'  CE outside mask (visible)  : {_fmt(ce_outside)}')
    print(f'  CE inside  mask (occluded) : {_fmt(ce_inside)}')